# Mise à jour des connexions des datasets d'un projet

Ce notebook identifie tous les datasets d'un projet dont la connexion contient `cos` ou `pg` dans son nom,
et les remplace par les nouvelles connexions cibles.

**Paramètre `DRY_RUN`** :
- `True` → affiche ce qui serait changé, sans modifier
- `False` → applique les modifications

In [ ]:
import dataikuapi
import dataiku
import re
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## Paramètres à renseigner

In [ ]:
# --- Paramètres obligatoires ---
PROJECT_KEY        = "MON_PROJET"        # Clé du projet Dataiku cible
NEW_COS_CONNECTION = "s3-nouvelle-cos"   # Nom exact de la nouvelle connexion COS
NEW_PG_CONNECTION  = "db-nouvelle-pg"    # Nom exact de la nouvelle connexion PostgreSQL

# --- Mode dry run ---
DRY_RUN = True   # True = simulation, False = application réelle

# --- Connexion au node (laisser vide si exécuté directement dans Dataiku) ---
DSS_HOST    = ""   # Ex: "https://mon-node.example.com" — laisser vide si exécuté dans Dataiku
DSS_API_KEY = ""   # Laisser vide si exécuté dans Dataiku

# --- Expressions régulières d'identification ---
COS_PATTERN = re.compile(r"cos", re.IGNORECASE)
PG_PATTERN  = re.compile(r"pg",  re.IGNORECASE)

## Connexion au client Dataiku

In [ ]:
if DSS_HOST and DSS_API_KEY:
    client = dataikuapi.DSSClient(DSS_HOST, DSS_API_KEY, insecure_tls=True)
    print(f"Connexion externe : {DSS_HOST}")
else:
    client = dataiku.api_client()
    print("Connexion via le client interne Dataiku")

project = client.get_project(PROJECT_KEY)
print(f"Projet : {PROJECT_KEY}")

## Scan des datasets et identification des connexions à changer

In [ ]:
datasets_to_update = []  # [(dataset_name, current_connection, new_connection)]
datasets_skipped   = []  # datasets sans connexion reconnue

all_datasets = project.list_datasets()
print(f"{len(all_datasets)} dataset(s) trouvé(s) dans le projet '{PROJECT_KEY}'\n")

for ds_summary in all_datasets:
    ds_name = ds_summary["name"]
    try:
        ds       = project.get_dataset(ds_name)
        ds_def   = ds.get_settings().get_raw()
        current_connection = ds_def.get("params", {}).get("connection", "")

        if not current_connection:
            datasets_skipped.append((ds_name, "pas de connexion"))
            continue

        if COS_PATTERN.search(current_connection):
            datasets_to_update.append((ds_name, current_connection, NEW_COS_CONNECTION))
        elif PG_PATTERN.search(current_connection):
            datasets_to_update.append((ds_name, current_connection, NEW_PG_CONNECTION))
        else:
            datasets_skipped.append((ds_name, current_connection))

    except Exception as e:
        print(f"  [ERREUR] {ds_name} : {e}")

print(f"{len(datasets_to_update)} dataset(s) à mettre à jour")
print(f"{len(datasets_skipped)} dataset(s) ignoré(s) (connexion non COS/PG ou absente)")

## Résumé des changements prévus

In [ ]:
if not datasets_to_update:
    print("Aucun dataset à mettre à jour.")
else:
    print(f"{'DATASET':<40} {'CONNEXION ACTUELLE':<35} {'NOUVELLE CONNEXION':<35}")
    print("-" * 110)
    for ds_name, current_conn, new_conn in datasets_to_update:
        changed = "[IDENTIQUE]" if current_conn == new_conn else ""
        print(f"{ds_name:<40} {current_conn:<35} {new_conn:<35} {changed}")

print()
if DRY_RUN:
    print("MODE DRY RUN activé — aucune modification ne sera appliquée.")
else:
    print("MODE RÉEL — les modifications vont être appliquées.")

## Application des modifications

In [ ]:
if DRY_RUN:
    print("DRY_RUN=True → aucune modification effectuée. Passez DRY_RUN=False pour appliquer.")
else:
    if not datasets_to_update:
        print("Aucun dataset à mettre à jour.")
    else:
        success_count = 0
        error_count   = 0

        for ds_name, current_conn, new_conn in datasets_to_update:
            if current_conn == new_conn:
                print(f"  [SKIP] {ds_name} — connexion déjà correcte : {current_conn}")
                continue
            try:
                ds       = project.get_dataset(ds_name)
                ds_settings = ds.get_settings()
                raw      = ds_settings.get_raw()

                raw["params"]["connection"] = new_conn
                ds_settings.save()

                print(f"  [OK] {ds_name} : '{current_conn}' → '{new_conn}'")
                success_count += 1
            except Exception as e:
                print(f"  [ERREUR] {ds_name} : {e}")
                error_count += 1

        print(f"\nTerminé — {success_count} mis à jour, {error_count} en erreur.")